# 🔐 Complete Guide to Encoding & Decoding Methods in Python

This notebook covers all major encoding/decoding techniques:
- What they are
- How they work (with code)
- Advantages & Disadvantages
- Where they work best

---

## 📦 Imports

In [ ]:
import base64
import binascii
import codecs
import hashlib
import html
import json
import pickle
import struct
import urllib.parse
import zlib
import gzip
import io
import sys

print('All standard-library modules imported successfully.')
print(f'Python version: {sys.version}')

---
## 1. 🔡 Character Encoding (UTF-8, UTF-16, ASCII, Latin-1)

**What it is:** Converts text (strings) into raw bytes using a defined mapping between characters and byte sequences.

| Encoding | Bits/char | Characters supported |
|----------|-----------|---------------------|
| ASCII | 7-bit | 128 (English only) |
| Latin-1 | 8-bit | 256 (Western European) |
| UTF-8 | 8–32-bit | All Unicode (variable length) |
| UTF-16 | 16–32-bit | All Unicode (fixed 2 or 4 bytes) |
| UTF-32 | 32-bit | All Unicode (fixed 4 bytes) |

**✅ Advantages:**
- UTF-8 is backward-compatible with ASCII
- Universal — can represent every language
- UTF-8 is space-efficient for ASCII-heavy text

**❌ Disadvantages:**
- UTF-16/32 waste space for ASCII text
- Mixing encodings causes `UnicodeDecodeError`
- ASCII cannot represent non-English characters

**📍 Best used:** File I/O, network transmission, databases — everywhere text is stored or sent.

In [ ]:
text = 'Hello, 世界! 🌍'

for enc in ['utf-8', 'utf-16', 'utf-32', 'latin-1']:
    try:
        encoded = text.encode(enc)
        decoded = encoded.decode(enc)
        print(f'{enc.upper():10s} | bytes={len(encoded):3d} | hex={encoded.hex()[:40]}...')
        print(f'           | decoded: "{decoded}"')
    except UnicodeEncodeError as e:
        print(f'{enc.upper():10s} | ERROR: {e}')
    print()

---
## 2. 🔢 Base64 Encoding

**What it is:** Converts binary data into a text-safe string using 64 printable ASCII characters (`A-Z`, `a-z`, `0-9`, `+`, `/`). Every 3 bytes → 4 characters.

**Process:**
1. Take 3 bytes (24 bits)
2. Split into four 6-bit groups
3. Map each 6-bit group to one of 64 characters
4. Pad with `=` if the input isn't a multiple of 3 bytes

**✅ Advantages:**
- Safe to transmit binary data over text protocols (email, JSON, XML)
- Widely supported across all languages and platforms
- Reversible (lossless)

**❌ Disadvantages:**
- ~33% size overhead
- Not encryption — anyone can decode it
- Not suitable for large binary files (bloats size)

**📍 Best used:** Email attachments (MIME), embedding images in HTML/CSS, JWT tokens, API payloads.

In [ ]:
# Standard Base64
original = b'Python encoding is powerful! \x00\x01\x02'

encoded = base64.b64encode(original)
decoded = base64.b64decode(encoded)

print('=== Standard Base64 ===')
print(f'Original  ({len(original)} bytes): {original}')
print(f'Encoded   ({len(encoded)} chars):  {encoded.decode()}')
print(f'Decoded:  {decoded}')
print(f'Match: {original == decoded}')

print()

# URL-safe Base64 (replaces + with - and / with _)
url_encoded = base64.urlsafe_b64encode(original)
url_decoded = base64.urlsafe_b64decode(url_encoded)

print('=== URL-Safe Base64 ===')
print(f'Encoded: {url_encoded.decode()}')
print(f'Match:   {original == url_decoded}')

print()
# Manual bit-level demo for 3 bytes
demo = b'Man'
bits = ''.join(f'{b:08b}' for b in demo)
groups = [bits[i:i+6] for i in range(0, 24, 6)]
alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'
chars = [alphabet[int(g, 2)] for g in groups]
print('=== Manual Bit Demo for "Man" ===')
print(f'Bytes:   {[hex(b) for b in demo]}')
print(f'Bits:    {bits}')
print(f'Groups:  {groups} → {[int(g,2) for g in groups]}')
print(f'Chars:   {chars} → "{"".join(chars)}"')
print(f'base64:  {base64.b64encode(demo)}')

---
## 3. 🔠 Hex (Base16) Encoding

**What it is:** Represents each byte as two hexadecimal digits (0–9, A–F). 1 byte → 2 characters.

**Process:** Each byte (0–255) is split into two nibbles (4 bits each), mapped to 0–F.

**✅ Advantages:**
- Human-readable representation of binary
- Easy to inspect bytes manually
- Used in debugging, cryptography output

**❌ Disadvantages:**
- 100% size overhead (doubles the data size)
- Not as compact as Base64

**📍 Best used:** Displaying cryptographic hashes, memory dumps, color codes (#FF5733), network packet inspection.

In [ ]:
data = b'Hello World!'

# Encode
hex_encoded = data.hex()           # bytes → hex string
hex_upper = binascii.hexlify(data) # bytes → bytes of hex

# Decode
decoded_1 = bytes.fromhex(hex_encoded)
decoded_2 = binascii.unhexlify(hex_upper)

print('=== Hex (Base16) Encoding ===')
print(f'Original:    {data}')
print(f'Hex string:  {hex_encoded}')
print(f'Hex bytes:   {hex_upper}')
print(f'Decoded:     {decoded_1}')
print(f'Match:       {data == decoded_1}')

print()
print('=== Byte-by-byte breakdown ===')
for char in data:
    print(f'  {chr(char)!r}  → decimal {char:3d} → binary {char:08b} → hex {char:02x}')

---
## 4. 🌐 URL Encoding (Percent Encoding)

**What it is:** Encodes special characters in URLs by replacing them with `%XX` where XX is the hex value of the character's UTF-8 byte.

**Process:** Any character not in the "unreserved" set (A–Z, a–z, 0–9, `-`, `_`, `.`, `~`) is encoded as `%HH`.

**✅ Advantages:**
- Makes arbitrary strings safe to use in URLs
- Universally supported by browsers and servers
- Handles all Unicode via UTF-8 byte encoding

**❌ Disadvantages:**
- Makes URLs long and unreadable
- Spaces become `%20` or `+` (inconsistent between `quote` vs `urlencode`)

**📍 Best used:** HTTP query strings, form submissions, REST API parameters.

In [ ]:
# Various strings to URL-encode
examples = [
    'Hello World',
    'search?q=Python & encoding=UTF-8',
    'price: $99.99 (discount 20%)',
    'email@example.com',
    'こんにちは',  # Japanese
]

print('=== URL Encoding (urllib.parse.quote) ===')
for s in examples:
    encoded = urllib.parse.quote(s)
    decoded = urllib.parse.unquote(encoded)
    print(f'Original: {s}')
    print(f'Encoded:  {encoded}')
    print(f'Decoded:  {decoded}')
    print()

print('=== URL Query String (urlencode) ===')
params = {'name': 'John Doe', 'city': 'New York', 'lang': 'Python & Go'}
encoded_qs = urllib.parse.urlencode(params)
decoded_qs = urllib.parse.parse_qs(encoded_qs)
print(f'Params:   {params}')
print(f'Encoded:  {encoded_qs}')
print(f'Decoded:  {decoded_qs}')

---
## 5. 📄 HTML Entity Encoding

**What it is:** Replaces HTML special characters (`<`, `>`, `&`, `"`, `'`) with their safe entity equivalents (`&lt;`, `&gt;`, `&amp;`, etc.).

**Process:** Scan string for characters with special HTML meaning → replace with `&name;` or `&#num;` entities.

**✅ Advantages:**
- Prevents XSS (Cross-Site Scripting) attacks
- Safely embeds user content into HTML
- Browser renders entities correctly

**❌ Disadvantages:**
- Increases string length
- Only needed in HTML context
- Must be decoded before string comparison

**📍 Best used:** Web templating, sanitizing user input before rendering in browsers, email HTML bodies.

In [ ]:
dangerous_strings = [
    '<script>alert("XSS")</script>',
    'Tom & Jerry > Spike & Tyke',
    '"Hello" & \'World\'',
    '<img src=x onerror=alert(1)>',
]

print('=== HTML Entity Encoding ===')
for s in dangerous_strings:
    escaped = html.escape(s, quote=True)  # also escapes quotes
    unescaped = html.unescape(escaped)
    print(f'Original:  {s}')
    print(f'Escaped:   {escaped}')
    print(f'Unescaped: {unescaped}')
    print(f'Match:     {s == unescaped}')
    print()

---
## 6. 📦 JSON Encoding/Decoding

**What it is:** Serializes Python objects (dicts, lists, strings, numbers, booleans, None) into a JSON string, and deserializes back.

**Process:** Python object → JSON text (encode). JSON text → Python object (decode).

**Type mapping:**
| Python | JSON |
|--------|------|
| dict | object `{}` |
| list/tuple | array `[]` |
| str | string `""` |
| int/float | number |
| True/False | true/false |
| None | null |

**✅ Advantages:**
- Human-readable
- Language-agnostic (works with any language)
- Native browser support (JavaScript)

**❌ Disadvantages:**
- Cannot serialize complex Python objects (custom classes, sets, datetimes) by default
- Less efficient than binary formats (MessagePack, Protocol Buffers)
- No type for date/time

**📍 Best used:** REST APIs, configuration files, data interchange between services.

In [ ]:
import json
import datetime

# Basic encoding/decoding
data = {
    'name': 'Alice',
    'age': 30,
    'scores': [95.5, 87.0, 92.3],
    'active': True,
    'address': None,
    'metadata': {'role': 'admin', 'level': 5}
}

json_str = json.dumps(data, indent=2)
recovered = json.loads(json_str)

print('=== JSON Encoding ===')
print(json_str)
print(f'\nMatch: {data == recovered}')

# Custom encoder for non-serializable types
class CustomEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime.datetime):
            return obj.isoformat()
        if isinstance(obj, set):
            return list(obj)
        return super().default(obj)

complex_data = {
    'timestamp': datetime.datetime(2024, 6, 15, 10, 30),
    'tags': {'python', 'encoding', 'data'},
}

print('\n=== Custom JSON Encoder ===')
print(json.dumps(complex_data, cls=CustomEncoder, indent=2))

---
## 7. 🥒 Pickle (Python Object Serialization)

**What it is:** Python-specific binary serialization of **any** Python object — including custom classes, functions, lambdas, numpy arrays, etc.

**Process:** Traverses the Python object graph and writes a binary bytecode stream that can reconstruct the exact object later.

**✅ Advantages:**
- Can serialize almost any Python object
- Preserves Python-specific types (tuples vs lists, etc.)
- Compact binary format (faster than JSON for complex objects)

**❌ Disadvantages:**
- Python-only (not cross-language)
- **Security risk** — never unpickle untrusted data (can execute arbitrary code)
- Not human-readable
- Version compatibility issues between Python versions

**📍 Best used:** ML model saving (scikit-learn), caching Python objects, inter-process communication within trusted Python systems.

In [ ]:
import pickle

class Person:
    def __init__(self, name, age, hobbies):
        self.name = name
        self.age = age
        self.hobbies = hobbies
    
    def greet(self):
        return f'Hi, I am {self.name}, {self.age} years old.'
    
    def __repr__(self):
        return f'Person(name={self.name!r}, age={self.age}, hobbies={self.hobbies})'

original = Person('Bob', 28, ['coding', 'chess', 'hiking'])

# Serialize to bytes
pickled = pickle.dumps(original, protocol=pickle.HIGHEST_PROTOCOL)
# Deserialize
restored = pickle.loads(pickled)

print('=== Pickle Serialization ===')
print(f'Original: {original}')
print(f'Pickled size: {len(pickled)} bytes')
print(f'Pickled (hex): {pickled.hex()[:60]}...')
print(f'Restored: {restored}')
print(f'Method works: {restored.greet()}')
print()

# Pickle to file
with open('/tmp/person.pkl', 'wb') as f:
    pickle.dump(original, f)

with open('/tmp/person.pkl', 'rb') as f:
    loaded = pickle.load(f)

print(f'File round-trip: {loaded}')
print(f'\n⚠️  WARNING: Never pickle.loads() data from untrusted sources!')

---
## 8. 🗜️ Compression Encoding (zlib / gzip)

**What it is:** Reduces data size using lossless compression algorithms (DEFLATE for zlib/gzip). Not encryption, but the output is binary and unreadable.

**Process (DEFLATE):**
1. **LZ77**: Replaces repeated byte sequences with back-references (offset + length)
2. **Huffman Coding**: Assigns shorter bit codes to more frequent symbols

| Format | Header | Use case |
|--------|--------|----------|
| zlib | zlib header + checksum | In-memory, network |
| gzip | gzip header + metadata | Files, HTTP Content-Encoding |

**✅ Advantages:**
- Significant size reduction (20–90% for text)
- Lossless — perfect reconstruction
- Very fast decompression

**❌ Disadvantages:**
- CPU cost to compress (less so to decompress)
- Little benefit on already-compressed data (jpg, mp4, zip)
- Not human-readable

**📍 Best used:** HTTP responses (gzip), log files, database BLOBs, network payloads, game assets.

In [ ]:
import zlib
import gzip
import io

# Repetitive text compresses very well
text = ('The quick brown fox jumps over the lazy dog. ' * 50).encode('utf-8')

print('=== zlib Compression ===')
for level in [1, 6, 9]:  # 1=fastest, 9=best compression
    compressed = zlib.compress(text, level=level)
    decompressed = zlib.decompress(compressed)
    ratio = len(compressed) / len(text) * 100
    print(f'Level {level}: {len(text)} → {len(compressed)} bytes ({ratio:.1f}%) | Match: {text == decompressed}')

print()
print('=== gzip Compression ===')
buf = io.BytesIO()
with gzip.GzipFile(fileobj=buf, mode='wb') as f:
    f.write(text)
gz_data = buf.getvalue()

buf2 = io.BytesIO(gz_data)
with gzip.GzipFile(fileobj=buf2, mode='rb') as f:
    restored = f.read()

print(f'Original:     {len(text)} bytes')
print(f'gzip output:  {len(gz_data)} bytes ({len(gz_data)/len(text)*100:.1f}%)')
print(f'Restored:     {len(restored)} bytes | Match: {text == restored}')

---
## 9. #️⃣ Hash Encoding (One-Way / Non-reversible)

**What it is:** Transforms data of any size into a fixed-size digest. **One-way** — cannot be decoded.

**Common algorithms:**
| Algorithm | Output size | Speed | Security |
|-----------|-------------|-------|----------|
| MD5 | 128-bit (32 hex) | Fastest | Broken (don't use for security) |
| SHA-1 | 160-bit (40 hex) | Fast | Weak (deprecated) |
| SHA-256 | 256-bit (64 hex) | Medium | Strong |
| SHA-512 | 512-bit (128 hex) | Slower | Very strong |
| blake2b | Variable | Fastest | Strong |

**✅ Advantages:**
- Fixed output size regardless of input
- Deterministic: same input = same hash
- Tiny change in input → completely different hash (avalanche effect)
- Cannot be reversed

**❌ Disadvantages:**
- **Not encoding** in the traditional sense — no decoding possible
- MD5/SHA-1 are vulnerable to collisions
- Without salting, hashes are vulnerable to rainbow table attacks

**📍 Best used:** Password storage (use bcrypt/argon2 in production), file integrity checks, data deduplication, digital signatures.

In [ ]:
import hashlib

message = b'Hello, Python!'
message_alt = b'Hello, Python.'

print('=== Hash Comparison ===')
print(f'Input 1: {message}')
print(f'Input 2: {message_alt} (only period changed)')
print()

for algo in ['md5', 'sha1', 'sha256', 'sha512', 'blake2b']:
    h1 = hashlib.new(algo, message).hexdigest()
    h2 = hashlib.new(algo, message_alt).hexdigest()
    print(f'{algo.upper()}')
    print(f'  msg1: {h1[:64]}')
    print(f'  msg2: {h2[:64]}')
    print(f'  same? {h1 == h2}')
    print()

print('=== File Integrity (SHA-256) ===')
content = b'Important file content that must not be tampered with.'
checksum = hashlib.sha256(content).hexdigest()
print(f'Checksum: {checksum}')

# Verify
tampered = b'Important file content that must not be tampered With.'  # capital W
verify_ok = hashlib.sha256(content).hexdigest() == checksum
verify_tampered = hashlib.sha256(tampered).hexdigest() == checksum
print(f'Original verified: {verify_ok}')
print(f'Tampered verified: {verify_tampered}')

---
## 10. 🧬 Binary / Struct Encoding

**What it is:** Packs Python values (ints, floats, bools) into raw binary bytes according to a format string. Like a manual C struct.

**Format characters:** `i`=int32, `f`=float32, `d`=float64, `s`=string, `B`=unsigned byte, `?`=bool

**Endianness prefixes:** `>` = big-endian, `<` = little-endian, `!` = network (big-endian)

**✅ Advantages:**
- Extremely compact (no overhead)
- Very fast — direct memory representation
- Compatible with C/C++ binary formats

**❌ Disadvantages:**
- Not human-readable
- Must know exact format to decode
- Endianness bugs across architectures

**📍 Best used:** Network protocols, file format parsing (PNG, BMP headers), inter-process communication, embedded systems, game binary files.

In [ ]:
import struct

# Pack a simple record: int, float, bool, 5-char string
fmt = '!i f ? 5s'  # big-endian: int32, float32, bool, 5-byte string
values = (42, 3.14159, True, b'hello')

packed = struct.pack(fmt, *values)
unpacked = struct.unpack(fmt, packed)

print('=== struct.pack / unpack ===')
print(f'Format:   {fmt}')
print(f'Values:   {values}')
print(f'Packed:   {packed.hex()}')
print(f'Size:     {len(packed)} bytes (vs ~30+ bytes in JSON)')
print(f'Unpacked: {unpacked}')

print()
print('=== Endianness Demo ===')
n = 0x01020304
big = struct.pack('>I', n)
little = struct.pack('<I', n)
print(f'Number: 0x{n:08X}')
print(f'Big-endian:    {big.hex()} (most significant byte first)')
print(f'Little-endian: {little.hex()} (least significant byte first)')

print()
print('=== Parsing a Fake PNG Header ===')
# PNG signature + IHDR chunk structure
width, height, bit_depth = 1920, 1080, 8
fake_ihdr = struct.pack('>II BB', width, height, bit_depth, 2)
w, h, bd, ct = struct.unpack('>II BB', fake_ihdr)
print(f'Width: {w}, Height: {h}, Bit depth: {bd}, Color type: {ct}')

---
## 11. 🔤 Codec Encodings (ROT13, bz2, hex_codec, etc.)

**What it is:** Python's `codecs` module provides many built-in codecs beyond character encoding — ROT13, bz2 compression, zlib, hex, base64, etc.

**✅ Advantages:**
- Unified API for many encode/decode operations
- Can chain codecs

**❌ Disadvantages:**
- ROT13 is trivially reversible — not for security
- Less explicit than using dedicated libraries

**📍 Best used:** Quick text transformations, obfuscation (not security), data format conversion.

In [ ]:
import codecs

text = 'Hello, World! Python is amazing.'

print('=== ROT13 (Caesar cipher shifted by 13) ===')
encoded = codecs.encode(text, 'rot_13')
decoded = codecs.decode(encoded, 'rot_13')  # same operation (symmetric)
print(f'Original: {text}')
print(f'ROT13:    {encoded}')
print(f'Decoded:  {decoded}')
print(f'Match:    {text == decoded}')

print()
print('=== bz2 via codecs ===')
data = b'Python encoding methods are very important for software development!'
bz2_encoded = codecs.encode(data, 'bz2_codec')
bz2_decoded = codecs.decode(bz2_encoded, 'bz2_codec')
print(f'Original: {len(data)} bytes')
print(f'bz2:      {len(bz2_encoded)} bytes')
print(f'Match:    {data == bz2_decoded}')

print()
print('=== hex_codec via codecs ===')
hex_enc = codecs.encode(b'hello', 'hex_codec')
hex_dec = codecs.decode(hex_enc, 'hex_codec')
print(f'hex: {hex_enc}  decoded: {hex_dec}')

---
## 12. 🔀 Base32 and Base85 Encoding

**Base32:** Uses 32 characters (A-Z, 2-7). Every 5 bytes → 8 characters. Case-insensitive.  
**Base85 (Ascii85):** Uses 85 printable ASCII chars. Every 4 bytes → 5 characters. More compact than Base64.

| Encoding | Chars used | Overhead | Use case |
|----------|-----------|----------|----------|
| Base16 | 16 | +100% | Hex dumps |
| Base32 | 32 | +60% | TOTP secrets, DNS labels |
| Base64 | 64 | +33% | General binary-to-text |
| Base85 | 85 | +25% | PDF, Git binary patches |

**📍 Best used:** Base32 → TOTP/2FA secret keys, case-insensitive contexts; Base85 → PDF streams, Git binary diffs.

In [ ]:
data = b'Binary data that needs text encoding: \x00\xFF\xAB'

b16 = base64.b16encode(data)
b32 = base64.b32encode(data)
b64 = base64.b64encode(data)
b85 = base64.b85encode(data)

print('=== Encoding Comparison ===')
print(f'Original ({len(data)} bytes): {data}')
print()
print(f'Base16 ({len(b16):3d} chars, +{(len(b16)/len(data)-1)*100:.0f}%): {b16.decode()}')
print(f'Base32 ({len(b32):3d} chars, +{(len(b32)/len(data)-1)*100:.0f}%): {b32.decode()}')
print(f'Base64 ({len(b64):3d} chars, +{(len(b64)/len(data)-1)*100:.0f}%): {b64.decode()}')
print(f'Base85 ({len(b85):3d} chars, +{(len(b85)/len(data)-1)*100:.0f}%): {b85.decode()}')

print()
print('=== Round-trip Verification ===')
print(f'b16 match: {base64.b16decode(b16) == data}')
print(f'b32 match: {base64.b32decode(b32) == data}')
print(f'b64 match: {base64.b64decode(b64) == data}')
print(f'b85 match: {base64.b85decode(b85) == data}')

---
## 📊 Summary Table: All Methods at a Glance

In [ ]:
summary = [
    ('Method',            'Reversible', 'Overhead',    'Readability', 'Best Use Case'),
    ('UTF-8',             '✅ Yes',      'Minimal',     'Human',       'Text files, web, databases'),
    ('ASCII',             '✅ Yes',      'Minimal',     'Human',       'English-only legacy systems'),
    ('Base64',            '✅ Yes',      '+33%',        'Not human',   'Email, JWT, API payloads'),
    ('Base32',            '✅ Yes',      '+60%',        'Not human',   'TOTP 2FA, case-insensitive'),
    ('Base85',            '✅ Yes',      '+25%',        'Not human',   'PDF streams, Git patches'),
    ('Hex/Base16',        '✅ Yes',      '+100%',       'Semi-human',  'Crypto output, debugging'),
    ('URL Encoding',      '✅ Yes',      'Variable',    'Semi-human',  'HTTP query params'),
    ('HTML Entities',     '✅ Yes',      'Small',       'Semi-human',  'XSS prevention, web content'),
    ('JSON',              '✅ Yes',      'Moderate',    'Human',       'REST APIs, config files'),
    ('Pickle',            '✅ Yes',      'Low',         'Not human',   'Python ML models, caching'),
    ('zlib/gzip',         '✅ Yes',      '-60 to -90%', 'Not human',   'HTTP compression, log files'),
    ('struct',            '✅ Yes',      'None',        'Not human',   'Binary protocols, file formats'),
    ('ROT13',             '✅ Yes',      'None',        'Obfuscated',  'Spoiler hiding, fun obfuscation'),
    ('SHA-256 (Hash)',     '❌ No',       'Fixed 32B',   'Not human',   'Integrity checks, passwords'),
    ('MD5 (Hash)',         '❌ No',       'Fixed 16B',   'Not human',   'Non-security checksums only'),
]

col_widths = [max(len(row[i]) for row in summary) for i in range(5)]

def row_str(row):
    return '  |  '.join(cell.ljust(col_widths[i]) for i, cell in enumerate(row))

separator = '-' * (sum(col_widths) + 15)
print(row_str(summary[0]))
print(separator)
for row in summary[1:]:
    print(row_str(row))

---
## 🧪 Practical: Layered Encoding Pipeline

Combining multiple encoding steps — common in real-world APIs and protocols.

In [ ]:
import json, zlib, base64, urllib.parse

# Simulate encoding a payload for a URL-safe, compressed API token
payload = {
    'user_id': 12345,
    'permissions': ['read', 'write', 'admin'],
    'expires': '2024-12-31T23:59:59',
    'metadata': 'A' * 200  # simulate larger payload
}

print('=== Layered Encoding Pipeline ===')
print('Step 1: Python dict → JSON')
step1 = json.dumps(payload).encode('utf-8')
print(f'  Size: {len(step1)} bytes')

print('Step 2: JSON → zlib compressed')
step2 = zlib.compress(step1, level=9)
print(f'  Size: {len(step2)} bytes ({len(step2)/len(step1)*100:.1f}%)')

print('Step 3: Compressed bytes → Base64 URL-safe')
step3 = base64.urlsafe_b64encode(step2).decode('ascii')
print(f'  Size: {len(step3)} chars')
print(f'  Token (first 80): {step3[:80]}...')

print('Step 4: URL-encode the token for embedding in a URL')
step4 = urllib.parse.quote(step3)
print(f'  URL: https://api.example.com/auth?token={step4[:60]}...')

print()
print('=== Decoding Pipeline (Reverse) ===')
d1 = urllib.parse.unquote(step4)
d2 = base64.urlsafe_b64decode(d1 + '==')
d3 = zlib.decompress(d2)
d4 = json.loads(d3.decode('utf-8'))
print(f'Recovered user_id: {d4["user_id"]}')
print(f'Recovered perms:   {d4["permissions"]}')
print(f'Full round-trip:   {d4 == payload}')

---
## ✅ Choosing the Right Encoding — Decision Guide

```
Is your data text?
  └─ Yes → Use UTF-8 (default), UTF-16 only for Windows/Java interop

Do you need binary → text conversion?
  ├─ Email/web/JWT → Base64
  ├─ URL-safe → Base64 URL-safe or Base85
  ├─ Case-insensitive/2FA → Base32
  └─ Debugging/hex display → Hex/Base16

Do you need data serialization?
  ├─ Cross-language / API → JSON
  ├─ Python-only / ML models → Pickle (trusted data only!)
  └─ Low-level binary protocol → struct

Do you need to reduce size?
  ├─ Text/HTML/JSON data → gzip or zlib
  └─ Already compressed (jpg/mp4) → Don't compress further

Do you need integrity verification?
  ├─ File checksums → SHA-256
  ├─ Password storage → bcrypt / argon2 (NOT MD5/SHA)
  └─ Non-security checksum → MD5 (fast)

Is it for a URL/browser?
  └─ URL Encoding (percent encoding)

Is it user content in HTML?
  └─ HTML Entity Encoding (prevent XSS)
```